
# Retention Regression Template

This notebook is a **template** for running the pair-specific Illinois-vs-control regression for the **Retention** analytic sample.

## What this notebook assumes

- You are running a regression for **one adjacent year pair at a time**
- Each row in the file already represents a **transition from year t to year t+1**
- The dependent variable is already constructed in the analytic sample:
  - `stayed_state_local`
- The key treatment indicator is:
  - `illinois` = 1 for Illinois, 0 for control states

## Interpretation

Because the file is already constructed at the transition level, this notebook estimates a **cross-sectional regression on a transition outcome** for a single year pair. This is your **pair-specific DiD-style regression** for the Retention margin.


In [ ]:
import pandas as pd
import numpy as np
import re
import statsmodels.api as sm
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 150)


#EDIT THIS when you switch to a new pair
DATA_FILE = "/mnt/data/analytic_sample_0809_retention.csv"

#Main variable choices
DV = "stayed_state_local"
TREATMENT_VAR = "illinois"
PAIR_VAR = "pair"
WEIGHT_VAR = "weight_t"

#Baseline controls
# If your file uses docc80_t instead of docc00_t, swap that one line below.
BASE_CONTROLS = [
    "age_t",
    "I(age_t**2)",
    "C(sex_t)",
    "C(race_t)",
    "C(grade92_t)",
    "np.log(earnwke_t)",
    "C(docc00_t)",
    "C(ind02_t)",
    "C(unionmme_t)",
]

#Robust SE choice
COV_TYPE = "HC1"


In [ ]:

#Load teh data
df = pd.read_csv(DATA_FILE, low_memory=False)

print("Shape:", df.shape)
print("\nPair values:")
print(df[PAIR_VAR].value_counts(dropna=False))

print("\nColumns:")
print(df.columns.tolist())


In [ ]:

#Data Validation
required_cols = [DV, TREATMENT_VAR, PAIR_VAR, WEIGHT_VAR]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Dataset is missing required columns: {missing}")

print("All required columns are present.")


In [ ]:
#Quick descriptive checks
print("\nUnweighted mean of DV by Illinois:")
print(df.groupby(TREATMENT_VAR)[DV].mean())

def weighted_mean(g):
    x = g[DV]
    w = g[WEIGHT_VAR]
    mask = x.notna() & w.notna()
    if mask.sum() == 0:
        return np.nan
    return np.average(x[mask], weights=w[mask])

print("\nWeighted mean of DV by Illinois:")
print(df.groupby(TREATMENT_VAR).apply(weighted_mean))

if "state_t" in df.columns:
    print("\nUnweighted mean of DV by state_t:")
    print(df.groupby("state_t")[DV].mean())



## Baseline specification

The default model in this notebook is a weighted Linear Probability Model:

$Y_i = \alpha + \beta \, \text{Illinois}_i + X_i'\theta + \varepsilon_i$

where:
- $Y_i$ is `stayed_state_local`
- the coefficient on `illinois` is the pair-specific Illinois-control difference for the **Retention** outcome


In [ ]:
def build_formula(dv, controls=None):
    controls = controls or []
    rhs_terms = [TREATMENT_VAR] + controls
    rhs = " + ".join(rhs_terms)
    return f"{dv} ~ {rhs}"

formula = build_formula(DV, BASE_CONTROLS)
print(formula)


In [ ]:
def extract_needed_columns(controls):
    """
    Pull raw dataframe column names out of patsy-style terms like:
    C(x), I(x**2), np.log(x), C(x):C(z), etc.
    """
    cols = set()

    for c in controls:
        c_matches = re.findall(r"C\(([^)]+)\)", c)
        for m in c_matches:
            cols.add(m.strip())

        i_matches = re.findall(r"I\(([^)]+)\)", c)
        for expr in i_matches:
            var_matches = re.findall(r"[A-Za-z_][A-Za-z0-9_]*", expr)
            for v in var_matches:
                if v not in {"I", "C", "np", "log"}:
                    cols.add(v)

        log_matches = re.findall(r"np\.log\(([^)]+)\)", c)
        for m in log_matches:
            cols.add(m.strip())

        if (
            not c.startswith("C(")
            and not c.startswith("I(")
            and not c.startswith("np.log(")
        ):
            var_matches = re.findall(r"[A-Za-z_][A-Za-z0-9_]*", c)
            for v in var_matches:
                if v not in {"I", "C", "np", "log"}:
                    cols.add(v)

    return list(cols)


def run_weighted_lpm(df, dv, controls=None, weight_var=WEIGHT_VAR, cov_type=COV_TYPE):
    controls = controls or []
    data = df.copy()

    raw_controls = extract_needed_columns(controls)

    needed_cols = [dv, TREATMENT_VAR, weight_var] + raw_controls
    needed_cols = [c for c in needed_cols if c in data.columns]

    before = len(data)
    data = data.dropna(subset=needed_cols).copy()

    if "np.log(earnwke_t)" in controls and "earnwke_t" in data.columns:
        data = data[data["earnwke_t"] > 0].copy()

    after = len(data)
    print(f"Dropped {before - after:,} rows due to missing values; {after:,} rows remain.")

    formula = build_formula(dv, controls)
    print(f"Running formula: {formula}")

    model = smf.wls(formula=formula, data=data, weights=data[weight_var])
    result = model.fit(cov_type=cov_type)

    print(result.summary())
    return result

lpm_result = run_weighted_lpm(df, DV, BASE_CONTROLS)


In [ ]:
result_table = pd.DataFrame({
    "term": ["illinois"],
    "coef": [lpm_result.params.get("illinois", np.nan)],
    "std_err": [lpm_result.bse.get("illinois", np.nan)],
    "p_value": [lpm_result.pvalues.get("illinois", np.nan)],
    "nobs": [int(lpm_result.nobs)]
})

result_table
